In [1]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mode, hmean, gmean, skew, kurtosis
c = "earthquake_dataset.csv"

df = pd.read_csv(c);

display(df)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def calculate_centrality_measures(data, measures):
    results = {}
    data = np.array(data)

    if 'Média' in measures:
        results['Média'] = np.mean(data)

    if 'Mediana' in measures:
        results['Mediana'] = np.median(data)

    if 'Moda' in measures:
        rounded_data = np.round(data)
        mode_result = mode(rounded_data, keepdims=True)
        if len(mode_result.count) > 0 and mode_result.count[0] > 1:
            results['Moda'] = mode_result.mode[0]
        else:
            results['Moda'] = 'Nenhuma moda definida'

    if 'Média Harmônica' in measures and np.all(data > 0):
        results['Média Harmônica'] = hmean(data[data > 0])

    if 'Média Geométrica' in measures and np.all(data > 0):
        results['Média Geométrica'] = gmean(data[data > 0])

    return results

def calculate_additional_measures(population):
    mean_value = np.mean(population)
    std_value = np.std(population)
    cv = (std_value / mean_value) * 100
    q1 = np.percentile(population, 25)
    q3 = np.percentile(population, 75)
    iqr = q3 - q1
    mad = np.median(np.abs(population - np.median(population)))

    print(f"Coeficiente de Variação (CV): {cv:.2f}%")
    print(f"Intervalo Interquartil (IQR): {iqr}")
    print(f"Desvio Absoluto Mediano (MAD): {mad}")

def analyze_data(data, title):
    skew_value = skew(data)
    kurt_value = kurtosis(data)
    std_value = np.std(data)
    var_value = np.var(data)

    print(f"{title}")
    print(f"Coeficiente de Assimetria: {skew_value:.2f}")
    print(f"Coeficiente de Curtose: {kurt_value:.2f}")
    print(f"Desvio Padrão: {std_value:.2f}")
    print(f"Variância: {var_value:.2f}")
    calculate_additional_measures(data)
    print("--------------------------------------------------")

def plot_distribution_side_by_side(population, measures, column_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(population, bins=30, kde=True, alpha=0.6, ax=axes[0])
    colors = ['red', 'green', 'purple', 'orange', 'brown']
    centrality_measures = calculate_centrality_measures(population, measures)

    print("Medidas de Centralidade Calculadas:")
    for measure, value in centrality_measures.items():
        print(f"{measure}: {value}")

    for i, (label, value) in enumerate(centrality_measures.items()):
        if isinstance(value, (int, float)):
            axes[0].axvline(value, color=colors[i % len(colors)], linestyle='--', label=f'{label}: {value:.2f}')

    axes[0].set_xlabel("Valor")
    axes[0].set_ylabel("Frequência")
    axes[0].set_title(f"Distribuição - {column_name}")
    axes[0].legend()

    sns.boxplot(x=population, ax=axes[1])
    axes[1].set_title(f'Boxplot - {column_name}')
    axes[1].set_xlabel('Valores')

    plt.tight_layout()
    plt.show()

# Lista de colunas para análise
selected_columns = ['Earthquake Magnitude', 'Depth (km)', 'Impact Score']

for column in selected_columns:
    if column in df.columns:
        data_series = df[column].dropna()
        data_array = data_series.to_numpy()
        analyze_data(data_array, f"Análise da Coluna: {column}")
        selected_measures = ['Média', 'Mediana', 'Moda', 'Média Harmônica', 'Média Geométrica']
        plot_distribution_side_by_side(data_array, selected_measures, column)